In [ ]:
import pandas as pd
import re
import os

# =============================================================================
# REPLACEMENT DICTIONARIES
# =============================================================================
# Each dictionary is applied at a different stage of variable name creation.
# All replacements are case-sensitive unless noted.
#
# HOW TO ADD:
#   "find this exact string": "replace with this"
#
# Keys are matched longest-first to avoid partial-word collisions.
# Empty string "" as a value means the key is simply deleted.
# =============================================================================


# -----------------------------------------------------------------------------
# STAGE 1 — Applied to raw Label text (before any other cleaning)
#
# Use this for human-readable label text transformations, e.g. expanding
# abbreviations or normalizing phrasing before the label becomes a varname.
# Matches are case-sensitive and applied to the original Label column.
#
# Example use cases:
#   Normalize punctuation:         "In the Past 12 Months" → ""
#   Standardize category names:    "Allocation Rate" → "Allocation/Imputation Rate ..."
# -----------------------------------------------------------------------------
LABEL_TEXT_REPLACEMENTS = {
    " in the Past 12 Months": "",                               # redundant for 1yr estimates
    "Allocation Rate": "Allocation/Imputation Rate (Imputation Rate of)",
    "Allocation of": "Allocation/Imputation (Imputation Rate of) ",
}


# -----------------------------------------------------------------------------
# STAGE 2 — Applied to label_varname and detail_varname (after lowercasing +
#           symbol-to-underscore conversion, before final cleanup)
#
# Use this for snake_case substring replacements — racial/ethnic group
# shorthands, redundant words, long phrases you want abbreviated, etc.
# All text here is already lowercase with spaces/symbols replaced by "_".
#
# Example use cases:
#   Shorten race/ethnicity labels: "black_or_african_american" → "black"
#   Remove filler words:           "_in_the_past_12_months"   → ""
#   Abbreviate common words:       "estimate" → "est"
# -----------------------------------------------------------------------------
VARNAME_REPLACEMENTS = {
    # Race / ethnicity
    "american_indian_and_alaska_native":          "AIAN",
    "black_or_african_american":                  "black",
    "native_hawaiian_and_other_pacific_islander": "NHPI",
    "some_other_race":                            "other_race",
    "two_or_more_races":                          "multiple_races",
    "not_hispanic_or_latino":                     "nh",
    "hispanic_or_latino":                         "hisp",
    "except_hispanic":                             "nh",
    "hispanic":                                   "hisp",
    "race_or_in_combination_with_one_or_more_other_races": "race_any",
    "_alone_or_in_combination_with_one_or_more_other_races": "_combo",
	"_alone_for_selected_tribal_ings":       "_tribal",
	"_alone_by_selected_":                   "_selected_",
	"_by_specific_origin":                   "_specific_origin",
	"_householder":                          "_hholder",  # for race-of-householder variants
    
    # Population scope suffixes (appear on nearly every label)
    "_for_the_civilian_noninstitutionalized_population": "_civ_noninst_pop",
    "_for_the_civilian_employed_population":             "_civ_emp_pop",
    "_for_the_civilian_population":                      "_civ_pop",
    "_for_the_population":                               "_pop",
    "_for_the_married_population":                       "_married_pop",
    "_for_workers":                                      "_workers",
    "_for_households":                                   "_hh",
    "_for_families":                                     "_fam",
    "_for_young_adults_aged_19_25":                      "_young_adults_19_25",

    # Age cutoff suffixes (redundant noise — the age is already in the varname)
    "_pop_16_yrs_over":   "_16plus",
    "_pop_15_yrs_over":   "_15plus",
    "_pop_25_yrs_over":   "_25plus",
    "_pop_18_yrs_over":   "_18plus",
    "_pop_65_yrs_over":   "_65plus",
    "_pop_3_yrs_over":    "_3plus",
    "_pop_5_yrs_over":    "_5plus",
    "_pop_16_64_yrs":     "_16_64",
    "_pop_18_64_yrs":     "_18_64",
    "_pop_25_64_yrs":     "_25_64",
    "_pop_1_yr_over":     "_1plus",
    "_60_yrs_over":        "_60plus",
    "_65_yrs_over":        "_65plus",
    "_30_yrs_over":        "_30plus",


    # Filler connector words
    "_of_the_householder": "_hholder",
    "_of_householder":     "_hholder",
    "_of_own_children":    "_own_child",
    "_of_related_children":"_rel_child",
    "_presence_of_own_children_under_18_yrs": "_own_child_u18",
    "_presence_of_related_children_under_18_yrs": "_rel_child_u18",
    "_under_18_yrs":       "_u18",
    "_under_18":           "_u18",

    # Redundant phrases or edited for conciseness
    "in_the_past_12_months":                     "", #(1yr ACS — these are always implied)
    "for_the_civilian_employed_population_16_yrs_and_over": "_civ_emp_pop16plus",
    "for_the_civilian_population_18_yrs_and_over":          "_civ_pop18plus",
    "for_the_population_16_yrs_and_over":                   "_pop16plus",
    "for_the_population_15_yrs_and_over":                   "_pop15plus",
    "for_the_foreign_born_population":                      "_foreign_born",
    "for_current_residence_in_the_united_states":           "",   # implied
    "in_the_united_states":                                 "",   # implied
    "in_puerto_rico":                                       "pr",
    "2022_inflation_adjusted_dollars":                      "2022dollars",
    "voting_age_population":                                "18plus",
    "_by_nativity_and_citizenship_status_by_sex": "_nativity_citizenship_sex",
    "_by_nativity_and_citizenship_status":        "_nativity_citizenship",
    "_by_marital_status":                         "_marital_status",
    "_by_age_by_":                               "_age_",
    "_by_age_":                                  "_age_",
    "ratio_of_income_poverty_level_of_families": "income_poverty_ratio_families",
    "poverty_status_of_families":                "poverty_families",
    "receipt_of_food_stamps_snap":               "snap_receipt",
    "geographical_mobility_in_the_past_yr":      "geo_mobility",
    "detailed_occupation_for_the":               "occupation_",
    "geographical_mobility_in_the_past_yr": "geo_mobility",
	"means_of_transportation_work":         "commute_mode",
	"time_of_departure_go_work":            "departure_time",
	"travel_time_work":                     "travel_time",
	"place_of_work":                        "POW",
	"language_spoken_at_home":              "lang_home",
	"ability_speak_english":                "eng_ability",
	"limited_english_speaking":             "limited_eng",
	"educational_attainment":               "educ_attain",
	"school_enrollment":                    "school_enroll",
	"household_income":                     "hh_income",
	"household_type":                       "hh_type",
	"household_size":                       "hh_size",
	"household_language":                   "hh_lang",
	"nonfamily_household":                  "nonfam_hh",
	"family_income":                        "fam_income",
	"family_type":                          "fam_type",
	"family_size":                          "fam_size",
	"marital_status":                       "marital_st",
	"employment_status":                    "emp_status",
	"work_experience":                      "work_exp",
	"disability_status":                    "disab_status",
	"veteran_status":                       "vet_status",
	"citizenship_status":                   "cit_status",
	"nativity_and_citizenship_status":      "nativity_cit",   # covers the combined form first
	"nativity_citizenship_status":          "nativity_cit",
	"health_insurance_coverage_status":     "hi_cov_status",
	"health_insurance_coverage":            "hi_cov",
	"private_health_insurance":             "private_hi",
	"public_health_insurance":              "public_hi",
	"ratio_of_income_poverty_level":        "inc_pov_ratio",
	"poverty_status":                       "pov_status",
	"supplemental_security_income_ssi":     "SSI",
	"supplemental_security_income":         "SSI",
	"social_security_income":               "SS_income",
	"public_assistance_income":             "pub_assist",
	"cash_public_assistance_income":        "cash_assist",
	"food_stamps_snap":                     "SNAP",
	"period_of_military_service":           "mil_service_period",
	"service_connected_disability_rating":  "svc_disab_rating",
	"selected_monthly_owner_costs":         "monthly_owner_costs",
	"gross_rent_as_a_percentage_of_household_income": "gross_rent_pct_income",
	"monthly_owner_costs_as_a_percentage_of_household_income": "owner_costs_pct_income",
	"housing_costs_as_a_percentage_of_household_income": "housing_costs_pct_income",
	"internet_subscription":                "internet_sub",
	"field_of_bachelor's_degree_for_first_major": "bachelors_field",
	"detailed_field_of_bachelor's_degree_for_first_major": "bachelors_field_detailed",
	"median_earnings_in_2022dollars":       "med_earnings",
	"household_income_in_2022dollars":      "hh_income",
	"family_income_in_2022dollars":         "fam_income",
	"earnings_in_2022dollars":              "earnings",
	"income_in_2022dollars":                "income",
	"in_2022dollars":                       "",            # strip remaining instances
	"_inflation_adjusted_dollars":          "",            # implied
	"full_time_yr_round":                   "FT_yr_round", # if not already handled
	"_for_workplace_geography":             "_workplace_geo",
	"metropolitan_statistical_area":        "metro_area",
	"micropolitan_statistical_area":        "micro_area",
	"not_metropolitan_or_micropolitan_statistical_area": "non_metro_micro_area",
	"state_county_place_level":             "state_county_place",
	"minor_civil_division_level":           "mcd_level",
	"living_arrangements":                  "living_arr",
	"grandparents_living_with_own_grandchildren": "gp_w_grandkids",
	"grandparents_responsible_for_own_grandchildren": "gp_resp_grandkids",
	"grandchildren_under_18_yrs_living_with_a_grandparent_householder": "grandkid_u18_gp_hholder",
	"_quarters_type_3_types":               "_GQ_3types",
	"_quarters_type_5_types":               "_GQ_5types",
	"_quarters_population":                 "_GQ_pop",
	"_quarters":                            "_GQ",
	"workers'_earnings":                    "worker_earnings",
	"number_of_workers_in":                 "n_workers_in",
	"number_of_earners_in":                 "n_earners_in",
	"number_of_persons_in":                 "n_persons_in",
	"number_of_times_married":              "n_times_married",
	"number_of_disabilities":               "n_disabilities",
	"_imputation_rate_of":                  "_imp_",        # collapses "imp_imputation_rate_of_X" → "imp_X"
	"imputation_rate_of":                   "imp_",
	"women_15_50_yrs_who_had_a_birth":      "women_15_50_birth",
	"women_16_50_yrs_who_had_a_birth":      "women_16_50_birth",
	"_for_current_residence":               "",            # implied for mobility tables
	"_for_the":                             "_",           # catch-all for remaining "for the" fragments
	"_of_the_":                             "_",           # catch-all

    # Symbols that slip through
    "$":                                          "",
    "_to_":                                       "_",        # e.g. age ranges "25_to_34" → "25_34"

    # Allocation/imputation shorthand (matches post-LABEL_TEXT_REPLACEMENTS output)
    "allocation_imputation_rate":                 "imp_rate",
    "allocation_imputation_of":                   "imp_of",
    "allocation_imputation":                      "imp",


    # Detail-level redundant phrases:
    "now_married_married_spouse_absent":               "married_spouse_absent",  # deduplicate
    "no_related_children_of_the_householder":          "no_related_children",
    "household_did_not_receive_food_stamps_snap":      "no_snap",
    "income_below_poverty_level":                      "below_poverty",
    "naturalized_u.s._citizen":                        "naturalized_citizen",
    "in_labor_force":                                  "labor_force",
    "with_earnings":                                   "w_earnings",
    "without_social_security_income":                  "no_ss_income",
    "with_ssi_and_or_cash_public_assistance_income":   "w_ssi_cash_assist",
    "with_ssi_and_cash_public_assistance_income":      "w_ssi_cash_assist",
    "male_householder_no_spouse_present":              "male_hholder_no_spouse",
    "female_householder_no_spouse_present":            "female_hholder_no_spouse",
    "american_indian_tribes_specified":                "AI_tribes",
    "entered_":                                        "entry_",   # e.g. "entry_2000_2009"
    "groups_tallied":                                  "groups",

    # Common word abbreviations
    "united_states":                                   "US",
    "estimate":                                   "", #default is that it's estimate
    "total":                                      "", #default is total
    "year":                                       "yr",
    "for_the_population":                         "pop",
    "groups":                                     "",
    "group":                                      "",
    "place_of_birth":                            "POB",
    "_and_or_":                                  "_or_",
    "_and_":                                     "_",  # catch-all, apply last
    "_yrs_and_over":                             "plus", #used for ages, like 15_yrs_and_older -> 15plus
    "_yrs_over":                                 "plus", #used for ages, like 15_yrs_and_older -> 15plus
    "full_time":                                    "FT",




}
# -----------------------------------------------------------------------------
# STAGE 3 — Applied to both_varname (the combined "label__detail" string)
#           after it has already been through STAGE 1 + STAGE 2 processing.
#
# Use this to clean up patterns that only appear once label and detail are
# joined, e.g. redundant prefixes introduced by a specific table's label.
#
# Example use cases:
#   Strip table-level prefix that adds noise to every variable name:
#       "acs_demographic_and_housing_ests__" → ""
# -----------------------------------------------------------------------------
BOTH_VARNAME_REPLACEMENTS = {
    "acs_demographic_and_housing_ests__": "",   # prefix redundant once variables are in context
    "selected_social_characteristics_in_puerto_rico__": "soc_chars_pr__",
    "selected_social_characteristics_in_the_united_states__": "soc_chars__",
    "selected_housing_characteristics__":              "housing_chars__",
    "selected_economic_characteristics__":             "econ_chars__",
    "population_60_yrs_and_over_in_the_united_states__": "pop60plus__",
}


# =============================================================================
# FILE PATHS  — change filenames here if needed
# =============================================================================
### !!! Keep these in sync with the filenames referenced in App.jsx !!! ###

wd         = os.getcwd()
DATA_DIR   = os.path.join(wd, "public")

INPUT_FILE  = "1yr_raw_variables.csv"
OUTPUT_FILE = "1yr_clean_varnames.csv"
OUTPUT_SHORT = "1yr_clean_varnames_shortened.csv"   # first 10,000 rows for dev/testing

inputFP      = os.path.join(DATA_DIR, INPUT_FILE)
outputFP     = os.path.join(DATA_DIR, OUTPUT_FILE)
outputFPshort = os.path.join(DATA_DIR, OUTPUT_SHORT)

### !!! ################################################################ !!! ###


# =============================================================================
# HELPERS
# =============================================================================

def _apply_dict(text: str, replacements: dict) -> str:
    """Apply a replacement dictionary to a string, longest keys first."""
    for key in sorted(replacements, key=len, reverse=True):
        text = text.replace(key, replacements[key])
    return text


def _to_snake(text: str) -> str:
    """Lowercase a string and convert common separators to underscores."""
    text = text.lower()
    for ch in ("!!", "(", ")", ",", ":", " ", "-", "/"):
        text = text.replace(ch, "_")
    # Collapse runs of underscores and strip leading/trailing ones
    text = re.sub(r"_+", "_", text).strip("_")
    return text


# =============================================================================
# CORE PROCESSING
# =============================================================================

def create_coding_varnames(csv_filepath: str, output_filepath: str = None) -> pd.DataFrame:
    df = pd.read_csv(csv_filepath)
    df["Label"]  = df["Label"].fillna("")
    df["Detail"] = df["Detail"].fillna("")
    df["Label_old"] = df["Label"]   # preserve original for reference

    # ------------------------------------------------------------------
    # Stage 1: clean label text (human-readable, pre-snake-case)
    # ------------------------------------------------------------------
    df["label_clean"] = df["Label"].apply(
        lambda text: _apply_dict(str(text), LABEL_TEXT_REPLACEMENTS)
    )

    # ------------------------------------------------------------------
    # Stage 2: convert to snake_case varnames, then apply VARNAME_REPLACEMENTS
    # ------------------------------------------------------------------
    df["label_varname"] = (
        df["label_clean"]
        .apply(_to_snake)
        .apply(lambda s: _apply_dict(s, VARNAME_REPLACEMENTS))
    )

    df["detail_varname"] = (
        df["Detail"]
        .apply(lambda text: _to_snake(str(text)))
        .apply(lambda s: _apply_dict(s, VARNAME_REPLACEMENTS))
    )

    # ------------------------------------------------------------------
    # Stage 3: combine into both_varname, then apply BOTH_VARNAME_REPLACEMENTS
    # ------------------------------------------------------------------
    df["both_varname"] = (
        (df["label_varname"] + "__" + df["detail_varname"])
        .apply(lambda s: _apply_dict(s, BOTH_VARNAME_REPLACEMENTS))
    )

    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Saved: {output_filepath}")

    return df


# =============================================================================
# RUN
# =============================================================================

if __name__ == "__main__":
    df_varnames = create_coding_varnames(inputFP, output_filepath=outputFP)
    df_varnames.head(10000).to_csv(outputFPshort, index=False)
    print(df_varnames.head(10).to_string())

Saved: /Users/klj9278/variable-explorer/public/1yr_clean_varnames.csv
            ID                                    Detail       Label Unnamed: 3 Unnamed: 4      Required                               Attributes Limit     Predicate Type   Group   Label_old label_clean label_varname      detail_varname                    both_varname
0       AIANHH                                 Geography                    NaN        NaN  not required                                      NaN     0  (not a predicate)     NaN                                                 geography                     __geography
1         ANRC                                 Geography                    NaN        NaN  not required                                      NaN     0  (not a predicate)     NaN                                                 geography                     __geography
2  B01001_001E                          Estimate!!Total:  Sex by Age        NaN        NaN  not required  B01001_001EA, B01